# 🧠 Tutorial 2 — EEG Preprocessing
###  · Biomedical Signals & Applications · EEG Python Workshop

---

**Goal of this tutorial:** Clean the raw EEG so it is ready for analysis.

**By the end you will be able to:**
- Re-reference EEG to the average reference
- Apply notch and bandpass filters
- Detect and interpolate bad channels
- Run ICA to remove eye-blink and muscle artefacts
- Cut the continuous recording into epochs around motor imagery cues
- Apply baseline correction and reject noisy trials
- Save the clean epochs for Tutorial 3

**Total time: ~3 hours** (including theory discussion and exercises)

---

> ⚠️ **Run Tutorial 1 first** — it downloads the dataset that this notebook uses.


---
## 🎓 Part 0 — Why Is Preprocessing Necessary?


> ⏱️ **Estimated time — Part 0 — Introduction: ~20 min**


### 0.1 What Does a Raw EEG Recording Look Like?

Imagine you place **64 small metal discs** (electrodes) on someone's scalp and record the tiny electrical signals produced by their brain for several minutes. What you get back is **not** a clean brain signal. Instead, you get a mixture of many things happening at the same time:

- The **brain signal** you actually want — faint oscillations (5–50 µV)
- **Eye blinks** — create a giant spike that can be **50 times bigger** than the brain signal
- **Jaw and forehead muscles twitching** — adds high-frequency noise
- **Heartbeat** — the electrical activity of the heart reaches the scalp too
- **50 Hz hum from power sockets** — couples into the recording cables
- **Electrode wobbling** when the person moves their head

> 💡 **Analogy — Noisy Café:** Think of it like trying to record someone **whispering** in a noisy café. There are other people talking, music playing, coffee machines running, and chairs scraping — all mixed into your microphone at the same time. Preprocessing is the process of cleaning up that recording so you can hear what the whispering person is actually saying.


### 0.2 How Big Are These Artefacts?

One of the most important things to understand is the **scale difference** between brain signals and noise. This is why preprocessing is not optional — it is essential.

| Source | Typical Amplitude | Compared to brain signal |
|--------|-------------------|--------------------------|
| **Brain EEG** (what we want) | 5 – 50 µV | × 1 — the target |
| Eye blink | 100 – 2000 µV | **× 10 to × 100 larger!** |
| Lateral eye movement | 50 – 500 µV | × 5 to × 50 larger |
| Jaw/forehead muscle | 50 – 1000 µV | × 5 to × 100 larger |
| Heartbeat (ECG) | 20 – 200 µV | × 2 to × 20 larger |
| 50 Hz power line | 5 – 100 µV | Similar to or larger than brain |
| Electrode movement | 100 – 5000 µV | Can completely mask the brain |

> 🎯 **Why This Matters:** Without preprocessing, any analysis you do will mostly reflect these artefacts — not the brain. A classifier trained on noisy EEG would learn to detect eye blinks, not motor imagery. Preprocessing is the **foundation** of everything else.

> 🗣️ **Discussion question for the class:** *"What would happen if we tried to classify left-hand vs right-hand motor imagery on raw EEG that still contains eye blinks?"*


### 0.3 The Preprocessing Pipeline — An Overview

We clean EEG in a **specific sequence** of steps. Each step targets a different type of noise, and **the order matters** (we explain why in Part 9). Here is the full pipeline we will follow today:

```
Load data
    │
    ▼
Set electrode positions (montage)
    │
    ▼
Mark & Interpolate bad channels  ←── must come FIRST
    │
    ▼
Average re-reference             ←── after bad channels fixed
    │
    ▼
Notch filter  (50 / 60 Hz)
    │
    ▼
Bandpass filter  (1 – 40 Hz)     ←── must come BEFORE ICA
    │
    ▼
ICA: fit → identify → remove     ←── must come BEFORE epoching
    │
    ▼
Epoch (cut into trials)
    │
    ▼
Baseline correction
    │
    ▼
Epoch rejection
    │
    ▼
Save clean epochs  →  Tutorial 3
```

> ⚠️ **The order is not arbitrary.** We will explain the three critical ordering rules at the end of this tutorial.


---
## ⚙️ Setup — Imports

Run this cell first every time you open the notebook.


> ⏱️ **Estimated time — Setup: ~5 min**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from mne.preprocessing import ICA

mne.set_log_level("WARNING")
print("MNE version:", mne.__version__)


---
## Step 0 — Load Raw Data


> ⏱️ **Estimated time — Step 0 — Load data: ~10 min**


### What dataset are we using?

We are using the **PhysioNet BCI Motor Movement/Imagery** dataset — one of the most widely-used public EEG datasets in research.

- **Subjects:** 109 healthy adults
- **Task:** The subject imagines moving their **left hand (T1)** or **right hand (T2)**
- **Channels:** 64 EEG electrodes arranged in the standard 10-05 system
- **Sampling rate:** 160 Hz (160 data points per second per channel)
- **Duration:** ~2.5 minutes per run (we load 2 runs)

The raw EEG object contains:
- **64 channels** × **~20,000 time points** = ~1.3 million numbers
- Two types of events: **T1** (left hand imagery) and **T2** (right hand imagery)
- Lots of noise — eye blinks, muscle tension, power-line interference

Our job in this tutorial is to clean all of that.

> 🗣️ **Discussion question:** *"With 64 channels and 160 samples/second, how many numbers per second are we recording? How many per minute?"*
>
> *Answer: 64 × 160 = 10,240 numbers/second = 614,400/minute*


In [ ]:
subject = 1
runs    = [6, 10]

raw_files = eegbci.load_data(subject, runs, verbose=False)
raw = concatenate_raws([read_raw_edf(f, preload=True, verbose=False)
                        for f in raw_files])
eegbci.standardize(raw)
raw.set_montage(mne.channels.make_standard_montage("standard_1005"), verbose=False)

print("Raw data loaded")
print(f"  Channels : {len(raw.ch_names)}")
print(f"  Duration : {raw.times[-1]:.1f} s")
print(f"  Sfreq    : {raw.info['sfreq']} Hz")


### Visualise the raw data

Let's look at 10 seconds of the raw EEG **before any cleaning**. You should be able to spot large spikes (eye blinks) and faster noise (muscle artefacts).


In [ ]:
# Show 10 seconds of raw EEG — look for obvious artefacts
fig, ax = plt.subplots(figsize=(14, 6))
data, times = raw.get_data(return_times=True)
t_mask = times < 10          # first 10 seconds
offset = np.arange(10) * 100e-6   # 100 µV spacing between channels

for i, ch in enumerate(raw.ch_names[:10]):
    ax.plot(times[t_mask], data[i, t_mask] * 1e6 + i * 100,
            lw=0.7, label=ch)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Channels (offset for display)")
ax.set_yticks(np.arange(10) * 100)
ax.set_yticklabels(raw.ch_names[:10])
ax.set_title("First 10 channels — first 10 seconds of RAW EEG (no cleaning yet)")
ax.set_xlim(0, 10)
plt.tight_layout()
plt.show()
print("\n👉 Can you spot any large spikes (eye blinks) or irregular bursts (muscle noise)?")


---
## Step 1 — Re-Referencing


> ⏱️ **Estimated time — Step 1 — Re-referencing: ~20 min**


### 1.1 What Is a Reference Electrode?

Before explaining re-referencing, we need to understand something fundamental about **voltage**.

> 💡 **Voltage is never absolute — it is always a difference between two points.**
>
> You cannot measure "the voltage at electrode Fp1". You can only measure "the voltage at Fp1 **compared to some other point**".

> 🗣️ **Analogy — Altitude:** Think of altitude. We say Mount Everest is 8,849 metres tall — but tall compared to *what*? Compared to sea level. If we chose the floor of the Dead Sea as our reference, Everest would have a different height number — but the mountain itself hasn't changed.
>
> Similarly, EEG measures brain voltage **compared to a reference point**. The brain hasn't changed — but the numbers you see depend entirely on where you placed the reference electrode.

In EEG, the amplifier records the voltage difference between each active electrode (on the scalp) and **one chosen reference electrode**. If that reference electrode happens to sit on a region with brain activity, that activity gets subtracted from every channel — distorting all recordings simultaneously.

**Example:** If the reference is placed at Cz (top of the head, directly over the motor cortex), then any motor cortex signal will be reduced in all other channels — because you are subtracting it out.


### 1.2 The Average Reference — Our Solution

The most popular solution for research EEG is the **average reference**. Instead of using one specific electrode as the reference, we use the **average of all electrodes** and subtract this average from every channel.

$$V_{\text{cleaned}}(\text{channel}) = V_{\text{recorded}}(\text{channel}) - \frac{1}{N}\sum_{k=1}^{N} V_{\text{recorded}}(k)$$

In plain words: for each time point, add up the voltage values from all electrodes, divide by the number of electrodes, and subtract that average from every channel.

> ⚠️ **Critical rule:** Always fix bad channels **before** computing the average reference. A broken channel recording noise at 500 µV would pull the average up by ~8 µV, corrupting every single channel.

> 🗣️ **Check your understanding:**
> 1. If one electrode is broken and recording 500 µV of noise, how does that affect the average reference? *(It inflates the average, which gets subtracted from every channel, corrupting all 63 other channels.)*
> 2. Why is the average reference better than a single reference electrode?


In [ ]:
raw_reref = raw.copy().set_eeg_reference("average", projection=False, verbose=False)
print("Re-referencing: set to average reference ✓")

# Visual comparison — one channel before vs after
data_before, times = raw.get_data(return_times=True)
data_after, _      = raw_reref.get_data(return_times=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(times[:1280], data_before[0, :1280] * 1e6, "steelblue", lw=0.8)
axes[0].set_title("Channel Fc5 — Before Re-referencing")
axes[0].set_ylabel("Amplitude (µV)")
axes[1].plot(times[:1280], data_after[0,  :1280] * 1e6, "tomato",    lw=0.8)
axes[1].set_title("Channel Fc5 — After Average Re-referencing")
axes[1].set_ylabel("Amplitude (µV)")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


> 👁️ **What to check:**
> - The two traces should look nearly identical for this dataset (it was already average-referenced during recording)
> - In real-world data with a biased reference you would see the DC level shift
> - The *shape* of the waveform should not change — only the overall level
> - In your own research: **always report which reference you used** in your methods section, as different choices produce visually different results

> ⚠️ **Note for European datasets:** The PhysioNet dataset was recorded in the USA. For European datasets (e.g., from BrainVision amplifiers in a Greek hospital), the same logic applies — you still use average reference. The only difference is the notch filter in the next step (50 Hz vs 60 Hz).


---
## Step 2 — Filtering


> ⏱️ **Estimated time — Step 2 — Filtering: ~30 min**


### 2.1 What Is a Frequency in EEG?

EEG signals are made up of **oscillations** — rhythmic up-and-down fluctuations. A 10 Hz signal goes up and down 10 times per second. A 50 Hz signal goes up and down 50 times per second.

Different types of **brain activity** produce oscillations at different speeds, and different types of **noise** also have characteristic frequencies.

> 💡 **Analogy — Music Equaliser:** Think of the equaliser on a music player — the device that lets you boost the bass or treble. It separates the audio into frequency bands and lets you control each one independently. A filter in EEG does exactly the same thing.

### 2.2 The EEG Frequency Bands

| Frequency Range | What Is There? | Action |
|---|---|---|
| **Below 1 Hz** | Electrode drift, sweat, slow body movements — NOT brain | Remove (high-pass filter) |
| **0.5 – 4 Hz** | **Delta** waves: deep sleep | Keep (for sleep studies) |
| **4 – 8 Hz** | **Theta** waves: memory, drowsiness | Keep |
| **8 – 13 Hz** | **Alpha** waves: relaxed wakefulness; **Mu** rhythm over motor cortex | Keep — critical for motor imagery |
| **13 – 30 Hz** | **Beta** waves: active thinking, motor planning | Keep — critical for motor imagery |
| **30 – 80 Hz** | **Gamma** waves: high-level cognition; also contains some EMG | Keep or partially remove |
| **Above 40–80 Hz** | Mainly muscle (EMG) noise — NOT brain | Remove (low-pass filter) |
| **50 Hz exactly** | Power socket hum — NOT brain | Remove (notch filter) |

> 🗣️ **Discussion question:** *"For a motor imagery BCI, which frequency bands are most important and why?"*
>
> *Expected answer: Alpha (8–13 Hz) and Beta (13–30 Hz). These contain the Mu rhythm and sensorimotor beta rhythms that change during imagined movement (Event-Related Desynchronisation).*


### 2.3 The Three Filters We Apply

**The Notch Filter — Removing the Power Socket Hum**

European power sockets supply electricity that oscillates at exactly **50 Hz** (60 Hz in USA). This frequency radiates from the cables in the walls and couples into EEG recording cables. A notch filter removes **one very narrow frequency** while leaving everything around it completely untouched.

> 💡 **Analogy — Radio interference:** Imagine a radio that receives a strong interference signal at exactly 100.0 MHz. A notch filter is like a tiny "hole" in the frequency response that blocks only that exact frequency. You can still hear everything else perfectly.

---

**The High-Pass Filter — Removing Slow Drifts**

Electrodes and skin produce very slow voltage changes caused by sweat, small movements, and temperature. These are so slow (below 1 Hz) that they look like long, smooth drifts in the signal.

> 💬 **In plain words:** High-pass means: *let high frequencies pass through; block low frequencies*. We set the cut-off at **1 Hz**. Anything slower than 1 cycle per second is removed. Brain alpha (10 Hz), beta (20 Hz), theta (6 Hz) — all safely above 1 Hz — pass through unchanged.

---

**The Low-Pass Filter — Removing Muscle Noise**

Muscles generate high-frequency electrical activity (EMG) when they contract. Jaw and forehead muscles are especially problematic because they sit close to EEG electrodes. Their noise is spread broadly across 20–300 Hz.

> 💬 **In plain words:** Low-pass means: *let low frequencies pass through; block high frequencies*. We set the cut-off at **40 Hz**. Everything above 40 Hz is removed.

---

> ⭐ **Remember:** High-pass + Low-pass together = **Bandpass filter**. Bandpass 1–40 Hz means: keep only the frequencies between 1 Hz and 40 Hz. This is the standard choice for **motor imagery** EEG analysis.
>
> ⚠️ **For ERP studies** (like the P300 or N400), use **0.1–40 Hz** instead — ERPs contain very slow components that would be removed by 1 Hz filtering.


### 2.4 Zero-Phase Filtering — Why It Matters

When you apply a filter, there is a risk of **shifting the signal in time** — making a brain response appear earlier or later than it actually was. This is called **phase distortion**. For ERP research, where timing is everything (a P300 at 300 ms has a different meaning from a P300 at 320 ms), phase distortion would corrupt all latency measurements.

MNE-Python applies filters in **zero-phase mode**, which means the filter is applied **twice**: once forward in time and once backward. The time shifts from the two passes cancel each other out exactly.

> ⭐ **Tip:** Zero-phase filtering is the **default** in MNE. You do not need to do anything special — it is automatic. Just be aware that this requires the full signal to be loaded into memory (`preload=True`), which is why we always set that option when loading EEG files.


In [ ]:
raw_filt = raw_reref.copy()

# 2a. Notch filter — remove power-line interference
# Change freqs=60 to freqs=50 for European datasets!
raw_filt.notch_filter(freqs=60, verbose=False)

# 2b. Bandpass filter — keep only 1–40 Hz
raw_filt.filter(l_freq=1.0, h_freq=40.0, verbose=False)

print("Filtering complete: notch at 60 Hz + bandpass 1–40 Hz ✓")
print()
print("Remember: for European recordings, use notch_filter(freqs=50)")
print("Remember: for ERP studies,        use filter(l_freq=0.1, h_freq=40)")


### Visualise — Power Spectral Density (PSD) Before vs After

The **Power Spectral Density (PSD)** shows how much power exists at each frequency. It is the best way to verify that filtering worked correctly.

> 💡 **Reading a PSD plot:**
> - The x-axis is frequency (Hz)
> - The y-axis is power (dB) — how much energy at that frequency (higher = more energy)
> - Normal EEG has a characteristic **1/f shape** — more power at lower frequencies, gradually decreasing
> - We will look for: the 60 Hz spike **before** filtering, and confirm it is **gone after**


In [ ]:
# Compare PSD before vs after filtering
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

raw_reref.compute_psd(fmax=80).plot(axes=axes[0], show=False)
axes[0].set_title("PSD — Before Filtering")
axes[0].axvline(60, color="red", lw=1.5, ls="--", label="60 Hz (power line)")
axes[0].legend(fontsize=9)

raw_filt.compute_psd(fmax=80).plot(axes=axes[1], show=False)
axes[1].set_title("PSD — After Filtering (notch 60 Hz + bandpass 1–40 Hz)")

plt.tight_layout()
plt.show()


> 👁️ **What to check in the PSD plots:**
> - **Before:** Large hump at low frequencies + a sharp spike at 60 Hz
> - **After:** Clean spectrum that drops off sharply above 40 Hz; the 60 Hz spike is gone
> - An **alpha peak (~10 Hz)** may be visible as a small bump in both plots
> - The cut-off at 40 Hz should be clearly visible — power drops almost to zero above it

> 🗣️ **Discussion question:** *"Look at the PSD before filtering. Where is the power concentrated? Which part of the spectrum might contain motor imagery information?"*


---
## Step 3 — Bad Channel Detection & Interpolation


> ⏱️ **Estimated time — Step 3 — Bad channels: ~20 min**


### 3.1 Why Do Channels Go Bad?

In an ideal world, every electrode would make perfect contact with the scalp and record only genuine brain signals. In practice, recordings are rarely perfect. Here are the most common reasons an electrode fails:

- **Poor gel contact:** not enough conductive gel between electrode and scalp → high impedance → the electrode picks up more noise than brain signal
- **Dry electrode:** gel dried out during a long recording session
- **Broken wire:** thin wire connecting electrode to amplifier is damaged → flat line or extremely noisy signal
- **Bridged electrodes:** two adjacent electrodes accidentally connected by too much gel → both channels record exactly the same signal
- **Saturated amplifier:** a very large movement pushed the amplifier beyond its input range → clipped flat line at maximum voltage

### 3.2 How to Identify a Bad Channel

| What you see | What it means |
|---|---|
| **Flat horizontal line** (zero or constant) | Broken wire, disconnected electrode, or saturated amplifier |
| **Extreme noise** — much higher than all other channels | Poor contact, high impedance |
| **Exact same signal** as an adjacent channel | Bridged electrodes |
| Regular spikes at heartbeat rate (~1 per second) | Electrode over a scalp vessel |
| Signal looks fine but **jumps suddenly** to a new level | Electrode pop — brief loss of contact, then recovers |

> 💡 **Analogy — Microphone Array:** Think of it like a microphone array in a recording studio. If one microphone has a broken cable, you do not throw away the whole recording. Instead, you flag that microphone, remove its broken track, and **reconstruct** what it should have recorded using the signals from the nearby microphones. That reconstruction is what spherical spline interpolation does for EEG.


### 3.3 Spherical Spline Interpolation — Rebuilding a Bad Channel

Once we have identified a bad channel, we do not simply delete it. Instead, we **reconstruct** a plausible estimate of what it should have recorded, using the signals from all the surrounding good electrodes.

> 💬 **In plain words:**
> 1. Take all the working electrodes around the bad one
> 2. Fit a **smooth surface** through their values at each time point *(think of it like stretching a smooth rubber sheet over the scalp, anchored to all the good electrode values)*
> 3. Read off the value of that smooth surface at the position of the bad electrode
>
> That reconstructed value replaces the broken channel's data.

> ⭐ **Note:** Spherical spline interpolation works well when only a **few isolated** electrodes are bad (up to about 5 out of 64). If more than 20% of channels are bad, the interpolation becomes unreliable — that recording may need to be excluded entirely.

> ⚠️ **Critical rule:** Always interpolate bad channels **BEFORE** computing the average reference. A bad channel contaminates the channel average and will introduce artefacts everywhere.


In [ ]:
# For this demo, we manually mark Fp1 as bad
# In a real session: visually inspect raw.plot() and mark the ones you find
raw_bad = raw_filt.copy()
raw_bad.info["bads"] = ["Fp1"]
print("Marked as bad:", raw_bad.info["bads"])

# Interpolate the bad channel from its neighbours
raw_interp = raw_bad.copy().interpolate_bads(reset_bads=True, verbose=False)
print("After interpolation — bad channels:", raw_interp.info["bads"])  # should be empty


In [ ]:
# Visual comparison: original vs bad (simulated) vs interpolated
idx_fp1 = raw_filt.ch_names.index("Fp1")
data_before, times = raw.get_data(return_times=True)
t_sel   = times[:640]  # first 4 seconds

data_orig   = raw_filt.get_data()[idx_fp1, :640]   * 1e6
data_interp = raw_interp.get_data()[idx_fp1, :640] * 1e6

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t_sel, data_orig,       "steelblue", lw=1, label="Original")
axes[0].set_title("Fp1 — Original signal (eye blink artefacts are normal here — they will be removed by ICA)")
axes[1].plot(t_sel, np.zeros(640),   "tomato",    lw=1, label="Zeroed (simulating dead channel)")
axes[1].set_title("Fp1 — Simulated dead channel (all zeros = what a flat-line electrode looks like)")
axes[2].plot(t_sel, data_interp,     "seagreen",  lw=1, label="Interpolated")
axes[2].set_title("Fp1 — Reconstructed from neighbouring electrodes (spherical spline interpolation)")

for ax in axes:
    ax.set_ylabel("µV")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
axes[2].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


> 👁️ **What to check:**
> - The **interpolated** channel (green) should look smoother than the original because it has no eye-blink spike — Fp1 is a frontal electrode that captures blinks, and its neighbours are slightly further from the eyes
> - The overall shape and scale should be similar to other nearby channels
> - If the interpolated channel looks completely flat or extremely noisy, there are likely no good neighbours (too many bad channels)

> 🗣️ **Practical tip:** In a real EEG study, always check electrode impedances **before** the recording starts and aim for < 20 kΩ. This prevents most bad channels before they happen.


---
## Step 4 — ICA: Removing Physiological Artefacts


> ⏱️ **Estimated time — Step 4 — ICA (longest section): ~40 min**


### 4.1 The Problem ICA Solves

Even after filtering, **eye blinks, eye movements, and cardiac artefacts** are still present in the EEG. These artefacts are **physiological** — they come from the body, not the environment — so no filter can remove them without also removing brain signals (because they overlap in frequency).

We need a smarter approach.

The challenge is that **every electrode records a mixture of many sources**: some brain regions, some eye muscles, some facial muscles, and the heart. We cannot separate them by looking at one electrode at a time, because all sources are mixed together everywhere.

We need to look at all 64 channels **simultaneously** and use the **spatial pattern** of the signal to separate the sources.


### 4.2 The Cocktail Party Analogy

> 💡 **Analogy:** Imagine you are at a party where **5 people are talking at the same time**. You place 5 microphones around the room. Each microphone records a **mixture** of all 5 voices — some voices louder, some quieter, depending on how far each microphone is from each speaker.
>
> Given only the 5 microphone recordings, can you recover the 5 original voices?
>
> **ICA does exactly this — but for EEG.**
>
> - The "microphones" are the EEG **electrodes**
> - The "voices" are the independent sources: **brain regions, eye muscles, heart**
> - ICA analyses all channels together and figures out the original sources

The key insight is that brain signals, eye blinks, and muscle activity are **statistically independent** — knowing when someone blinks tells you nothing about what their alpha rhythm is doing at that moment. ICA exploits this independence to disentangle them.


### 4.3 How ICA Works — Step by Step

ICA analyses the patterns of co-variation across all 64 channels and identifies a set of **independent components (ICs)**. Each IC has two representations:

1. A **topographic map** showing which electrodes "feel" this source the most (its spatial distribution on the scalp)
2. A **time series** showing *when* this source was active

After ICA, you look at each component and ask: *does this look like a brain source or an artefact?*

| Component type | What the scalp map looks like | What the time series looks like |
|---|---|---|
| **Eye blink** | Large activity over the forehead electrodes (Fp1, Fp2) — symmetric | Occasional large spikes — once every few seconds |
| **Eye movement** | Opposite activity at left-front vs right-front electrodes | Step-like shifts when the person moves their eyes |
| **Heartbeat** | Scattered activity; often stronger at temples | Regular sharp pulses — about once per second |
| **Jaw muscle** | Irregular patches over temporal/cheek electrodes | Bursts of high-frequency noise when tense |
| **Brain — alpha** | Smooth, symmetric blob over the **back** of the head | Waxing and waning 10 Hz oscillation when relaxed |
| **Brain — motor mu** | Smooth blob centred over C3 or C4 | Rhythmic 10 Hz oscillation that disappears during movement |

> ⭐ **Remember:** ICA requires a **good amount of data** to work well. That is why we apply ICA to the **full continuous recording** (before epoching), not to individual short trials. More data = better source separation. ICA also requires that the data has been **high-pass filtered at ≥ 1 Hz** first — slow drifts confuse the ICA algorithm.

> ⚠️ **Golden rule:** When in doubt — **keep** the component. Removing a brain component permanently deletes neural information. Typically remove only **1–3 components** (eye blinks + possibly heartbeat).


### 4.4 Fitting the ICA Model

We fit ICA on the **filtered** data. This learns 20 independent components from the continuous EEG.

> 💬 **What `n_components=20` means:** We ask ICA to find 20 independent sources. With 64 channels, we could go up to 64 components, but 20 is enough to capture the main artefact sources while keeping computation fast. Each component will represent either a brain source or an artefact source.


In [ ]:
n_components = 20
ica = ICA(n_components=n_components, random_state=42, max_iter="auto")
ica.fit(raw_filt, verbose=False)

print(f"ICA fitted: {n_components} independent components extracted ✓")
print()
print("Next steps:")
print("  1. Plot component TOPOGRAPHIES → identify the spatial pattern of each source")
print("  2. Plot component TIME SERIES  → identify the temporal pattern (spikes = blinks)")
print("  3. Use automated detection    → MNE's find_bads_eog() as a starting point")
print("  4. ALWAYS visually verify     → never remove a component without inspection")


### 4.5 Inspect Component Topographies

Plot all 20 component maps. You are looking for characteristic patterns that indicate artefacts.

> 👁️ **What to look for:**
> - **Eye blink:** Large **symmetric blob** at the very front of the head (over Fp1/Fp2)
> - **Lateral eye movement:** **Asymmetric** front pattern — positive left, negative right (or vice versa)
> - **Muscle:** Focal spots at the **edges/bottom** of the scalp map
> - **Brain (alpha):** Smooth blob over the **back** of the head (occipital)
> - **Brain (motor mu):** Two symmetric spots over the **C3/C4** area

> 🗣️ **Class exercise:** After the plots appear, go through each component together. Call on students to identify which components look like artefacts and which look like brain activity.


In [ ]:
# Plot all component topographies — look for the frontal eye-blink blob
fig = ica.plot_components(show=True)
plt.suptitle("ICA Component Topographies\n"
             "Look for: frontal symmetric blob (eye blink), "
             "asymmetric front (eye movement), edge patches (muscle)", y=1.02)
plt.show()


### 4.6 Inspect Component Time Series

Plot the time series of each component. You are looking for:
- **Occasional large sharp spikes** → eye blinks
- **Regular pulses at ~1/second** → heartbeat
- **Bursts of fast oscillations** → muscle activity
- **Smooth 10 Hz oscillations** → alpha brain activity (keep this!)


In [ ]:
# Plot component time series — look for regular sharp triangular spikes (blinks)
fig = ica.plot_sources(raw_filt, show=True)
plt.show()


### 4.7 Automated Eye-Artefact Detection

MNE can help identify eye-movement components automatically by computing the **correlation** between each ICA component and a frontal channel (Fp1) that is close to the eyes. High correlation = likely eye artefact.

> ⚠️ **Important:** This is a **starting point**, not a final answer. Always visually verify the flagged components before removing them.


In [ ]:
# Automated eye artefact detection using Fp1 as a proxy for eye movements
eog_ch = "Fp1"
eog_idx, eog_scores = ica.find_bads_eog(raw_filt, ch_name=eog_ch, verbose=False)
print(f"Automated detection — suspected eye-movement components: {eog_idx}")
print()
print("Always verify these visually before removing!")

# Plot correlation scores
fig = ica.plot_scores(eog_scores, exclude=eog_idx, show=True,
                      title="ICA component correlation with Fp1 (eye proxy)\n"
                            "Bars in red are flagged as suspected eye artefacts")
plt.show()


In [ ]:
# Exclude the identified artefact components and reconstruct clean EEG
ica.exclude = eog_idx
raw_ica = raw_filt.copy()
ica.apply(raw_ica, verbose=False)

print(f"Artefact components removed: {ica.exclude}")
print(f"Clean data shape: {raw_ica.get_data().shape}")


> 👁️ **What to check after ICA:**
> - The large frontal spikes (eye blinks) should be **much smaller or gone**
> - The rest of the signal should look **broadly similar** — we only removed artefact sources
> - If the signal looks completely different, you may have removed **too many components**

> 🗣️ **Discussion question:** *"If we removed 3 components, does that mean we lost brain information?"*
>
> *Expected answer: Ideally no — if we identified them correctly as artefacts, we removed artefact sources. But if we accidentally removed a brain component, we have permanently lost that neural information. This is why visual inspection and the "when in doubt, keep it" rule are so important.*


---
## Step 5 — Epoching


> ⏱️ **Estimated time — Step 5 — Epoching: ~20 min**


### 5.1 From Continuous Recording to Trials

So far we have **one long continuous recording**. But our experiment had many individual trials — each time the subject received a cue (T1 or T2) to imagine moving their hand. **Epoching** cuts the continuous EEG into fixed-length windows aligned to each event.

> 💡 **Analogy — Cooking Video:** Imagine you are filming someone cooking throughout the day (continuous recording). Now you want to study what they do when a kitchen timer goes off (the event). You go through the footage and cut out a clip starting 30 seconds before each timer ring and ending 3 minutes after it. Each clip is an epoch. You then stack all the clips on top of each other and look for common patterns.

Each epoch runs from **−0.5 s** (before the cue, for baseline) to **+3.0 s** (end of the imagery period):

```
|-- baseline --|------------- imagery period ------------|
-0.5 s         0 s                                    +3.0 s
               ↑ cue appears here
```

After epoching, your data changes shape:
- **Before:** 2D array — (channels × all time samples)
- **After:** 3D array — (number of trials × channels × time samples per trial)

This 3D format is the standard input for all ERP and BCI classification analyses.


### 5.2 Baseline Correction

After cutting epochs, we **subtract the mean voltage** during the pre-cue period (−0.5 to 0 s) from the whole epoch. This zeros out any slow drift that happened to be present at the moment the cue appeared.

> 💡 **Analogy — Temperature Change:** Think of measuring how much the temperature changes after you turn on a heater. If you start measuring at 18°C in one room and 21°C in another, you cannot directly compare the measurements. But if you subtract the starting temperature from each measurement, you get "change from starting temperature" — which is directly comparable across rooms (and trials).

> 💬 **In plain words:**
> 1. Calculate the **average voltage** during the pre-stimulus period (−0.5 to 0 s)
> 2. Subtract that average from **every time point** in the epoch
>
> After this, every trial starts at exactly **zero** at the moment the stimulus appears. This makes trials comparable to each other regardless of what the brain was doing before.


In [ ]:
# Extract event markers (T1 = left hand, T2 = right hand)
events, event_id = mne.events_from_annotations(raw_ica, verbose=False)
print("Event IDs:", event_id)

# Keep only motor imagery events
event_id_sel = {"left_hand": event_id["T1"], "right_hand": event_id["T2"]}

tmin, tmax = -0.5, 3.0   # epoch window: 0.5 s baseline + 3.0 s imagery

epochs = mne.Epochs(
    raw_ica,
    events,
    event_id   = event_id_sel,
    tmin       = tmin,
    tmax       = tmax,
    baseline   = (None, 0),   # baseline-correct using the pre-stimulus interval
    preload    = True,
    verbose    = False,
)

print(f"\nEpochs created:")
print(f"  Trials    : {len(epochs)}")
print(f"  Shape     : {epochs.get_data().shape}  (trials × channels × time points)")
print(f"  Conditions: {list(epochs.event_id.keys())}")
print(f"  Time range: {tmin} s to {tmax} s")
print(f"  Baseline  : {tmin} s to 0 s (pre-cue period)")


In [ ]:
# Plot the first 5 epochs at channel C3
# C3 is over the LEFT motor cortex — important for right-hand motor imagery
ep_data  = epochs.get_data()
ep_times = epochs.times
idx_c3   = epochs.ch_names.index("C3")

fig, ax = plt.subplots(figsize=(11, 4))
for i in range(min(5, len(ep_data))):
    ax.plot(ep_times, ep_data[i, idx_c3] * 1e6, alpha=0.6, lw=0.9,
            label=f"Trial {i+1}")
ax.axvline(0, color="k", lw=1.5, linestyle="--", label="Cue onset (t=0)")
ax.axvspan(tmin, 0, alpha=0.08, color="grey", label="Baseline period")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title("C3 (left motor cortex) — First 5 Epochs\n"
             "Each line is one trial; cue appears at t=0; baseline period is grey")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Epoch image — rows=trials, columns=time, colour=amplitude
# This gives an overview of ALL trials at once — rows are trials, colour is amplitude
print("Epoch image for channel C3:")
print("  → Each row = one trial")
print("  → Colour   = voltage (blue = negative, red = positive)")
print("  → Look for patterns that appear consistently across rows (= real brain signal)")
print("  → Bright rows that look different from others (= possible artefacts to reject)")
fig = epochs.plot_image(picks=["C3"], show=True)
plt.show()


> 👁️ **What to check in the epoch image:**
> - Look for **consistent patterns** across rows — these represent reliable brain activity
> - Look for **bright or dark outlier rows** — these are likely artefacted trials that will be rejected
> - The **baseline period** (left side of plot, before t=0) should look relatively flat and consistent
> - Any trial that looks dramatically different from its neighbours is a candidate for rejection

> 🗣️ **Discussion question:** *"Why do we use the period BEFORE the cue as baseline? What would happen if we used the period AFTER the cue?"*
>
> *Answer: The pre-cue period should not be contaminated by the task-related brain response. Using the post-cue period as baseline would subtract the brain response we are trying to measure — destroying the signal of interest.*


---
## Step 6 — Epoch Rejection


> ⏱️ **Estimated time — Step 6 — Epoch rejection: ~10 min**


### 6.1 Why Some Trials Must Be Discarded

Despite all the preprocessing steps above, some individual trials will still contain large artefacts that could not be removed. This happens when:
- The participant moved their head suddenly — creating a huge transient movement artefact
- An electrode popped off and back on during a trial
- The participant sneezed, coughed, or laughed
- A very large muscle contraction overwhelmed the ICA decomposition

These trials are so contaminated that keeping them would corrupt the analysis.

### 6.2 The Amplitude Threshold Method

For each trial and each channel, we compute the **peak-to-peak amplitude** (max − min). If this exceeds a threshold that would be physiologically implausible for a clean EEG signal, we reject the entire trial.

| Threshold | Effect | When to use |
|---|---|---|
| **75 µV** | Very strict — rejects many trials | Only if you have > 100 trials and a very clean recording |
| **100 µV** | Strict | Standard for well-controlled recordings in cooperative adults |
| **150 µV** | Moderate — **recommended starting point** | Most EEG paradigms, motor imagery with 64 channels |
| **200 µV** | Lenient | When you cannot afford to lose trials (few per condition) |

> ⭐ **Target:** Aim to retain at least **70–80% of epochs**. If you are rejecting more than 30%, something in the earlier steps may need adjustment, or the recording quality was poor from the start.


In [ ]:
n_before = len(epochs)

epochs_clean = epochs.copy().drop_bad(
    reject={"eeg": 150e-6},   # ±150 µV threshold (150e-6 = 150 µV in Volts)
    verbose=False,
)

n_after  = len(epochs_clean)
retained = n_after / n_before * 100

print(f"Epochs before rejection : {n_before}")
print(f"Epochs after  rejection : {n_after}")
print(f"Retained                : {retained:.1f}%")
print()
if retained < 70:
    print("⚠️  Less than 70% retained — consider adjusting the threshold or reviewing ICA")
else:
    print("✓  Retention rate is acceptable")
print()
print(f"Left-hand trials  : {len(epochs_clean['left_hand'])}")
print(f"Right-hand trials : {len(epochs_clean['right_hand'])}")


### Visualise — Which Channels Caused the Most Rejections?


In [ ]:
# Show the drop log — what fraction of epochs was dropped and why
epochs_clean.plot_drop_log(show=True)
plt.show()


---
## Step 7 — Save Clean Epochs

We save the preprocessed epochs in MNE's `.fif` format. Tutorial 3 will load this file directly — no need to re-run all the preprocessing steps.


> ⏱️ **Estimated time — Step 7 — Save: ~5 min**


In [ ]:
# Apply explicit baseline correction (already applied in Epochs, but shown for clarity)
epochs_bc = epochs_clean.copy().apply_baseline((None, 0), verbose=False)

save_path = "/tmp/motor_imagery_epochs-epo.fif"
epochs_bc.save(save_path, overwrite=True, verbose=False)

print(f"Clean epochs saved to: {save_path} ✓")
print(f"Final shape: {epochs_bc.get_data().shape}  (trials × channels × time points)")
print()
# Verify we can load them back
epochs_loaded = mne.read_epochs(save_path, verbose=False)
print(f"Load check: {epochs_loaded.get_data().shape} ✓")
print()
print("These epochs will be the input for Tutorial 3: Feature Extraction & Classification")


---
## 🎓 Part 9 — Why the Order Matters

> 🗣️ *"Can I just do these steps in any order?"*
>
> **No.** The sequence is important because each step assumes the previous ones have already been done. Here are the three most important ordering rules.


### Rule 1: Fix Bad Channels BEFORE Computing the Average Reference

The average reference subtracts the **mean of all channels** from each channel. If one channel is broken and recording noise at 500 µV, that noise gets added (in negative) to every other channel. You must repair the broken channel first, so the average is meaningful.

> 💡 **Analogy:** If one person in a group is reporting absurd numbers, their number should be removed from the group average before you calculate it — not after.

---

### Rule 2: Apply Bandpass Filter BEFORE Running ICA

ICA is confused by slow drifts (below 1 Hz). Without high-pass filtering first, ICA tends to spend several of its components modelling the slow drift instead of the eye blinks and heartbeat. The result is poor artefact separation and components that are hard to interpret.

> 💡 **Analogy:** Before trying to separate voices at a cocktail party, you would turn off the air conditioning (the low-frequency hum). Otherwise the microphones capture that dominant rumble and the voice-separation algorithm tries to model it instead of the voices.

---

### Rule 3: Run ICA BEFORE Creating Epochs

ICA works better on **longer continuous data**. The more data it sees, the more accurately it estimates the spatial patterns of each source. Epoched data consists of many short segments with gaps between them, which disrupts the statistical estimation.

> 💡 **Analogy:** You would ask a music recogniser to listen to a full song before identifying the instruments — not to listen to thousands of disconnected one-second clips with silence in between.

---

> ⭐ **The correct order:**
> ```
> Load → Montage → Bad Channels → Average Reference → Notch Filter
> → Bandpass Filter → ICA → Epoch → Baseline Correction → Epoch Rejection → Save
> ```
> This order is used by research groups worldwide. Following it ensures your preprocessing decisions are scientifically defensible.

> 🗣️ **Final discussion questions:**
> 1. *"What would go wrong if we epoched BEFORE running ICA?"*
> 2. *"What would go wrong if we computed the average reference BEFORE fixing bad channels?"*
> 3. *"A student in a future project decides to bandpass filter their data to 0.1–40 Hz instead of 1–40 Hz. Is this necessarily wrong? When would it be appropriate?"*


---
## ✅ Tutorial 2 — Complete Summary

| Step | MNE Command | What it does | Why it matters |
|---|---|---|---|
| 1 — Re-reference | `set_eeg_reference('average')` | Remove reference electrode bias | All channels contaminated by reference choice |
| 2 — Notch filter | `notch_filter(freqs=60)` | Delete 60 Hz power-line hum | 60 Hz spike corrupts spectra and ICA |
| 2 — Bandpass | `filter(l_freq=1, h_freq=40)` | Keep only 1–40 Hz | Removes drifts (low end) and muscle noise (high end) |
| 3 — Bad channels | `info['bads'] = [...]` | Mark broken electrodes | Must be fixed before average reference |
| 3 — Interpolate | `interpolate_bads()` | Reconstruct from neighbours | Fills gaps without deleting spatial information |
| 4 — ICA fit | `ICA().fit(raw)` | Learn 20 independent components | Statistical learning requires full continuous data |
| 4 — ICA identify | `find_bads_eog()` + visual | Find artefact components | Automated + visual = safest approach |
| 4 — ICA apply | `ica.apply(raw)` | Remove artefacts; reconstruct EEG | Physiological artefacts can't be filtered — ICA is the solution |
| 5 — Epoch | `mne.Epochs(tmin, tmax, baseline)` | Cut into 3.5 s windows | Links brain signals to experimental events |
| 6 — Reject | `epochs.drop_bad(reject={...})` | Discard trials above threshold | Keeps data quality high; aim for ≥70% retention |
| 7 — Save | `epochs.save('-epo.fif')` | Store to disk for Tutorial 3 | Avoid repeating preprocessing every time |

---

> ⚠️ **Order matters!**
> `Fix bad channels → Re-reference → Filter → ICA → Epoch`
> Changing the order leads to worse artefact removal.

---

### 📋 Quick Reference Cheat Sheet

| Concept | One-sentence explanation | Key number / parameter |
|---|---|---|
| Average reference | Subtract mean of all channels from each channel | Requires ≥ 32 channels; do AFTER fixing bad channels |
| Notch filter | Removes exactly 50 Hz (or 60 Hz) mains interference | 50 Hz in Europe; 60 Hz in USA |
| High-pass filter | Removes everything slower than the cutoff | 1 Hz for motor imagery; 0.1 Hz for ERP studies |
| Low-pass filter | Removes everything faster than the cutoff | 40 Hz removes most EMG |
| Bandpass filter | High-pass + low-pass: keeps a frequency band | 1–40 Hz standard for motor imagery |
| Bad channel | Electrode not recording usable brain signals | Mark if flat, extreme noise, or identical to neighbour |
| Interpolation | Reconstruct bad channel from surrounding good channels | Works for up to ~5 bad channels out of 64 |
| ICA | Separate EEG into independent sources; remove artefact ones | 20 components; always visually verify before removing |
| Epoch | Fixed-length EEG window around an event marker | Typical: −0.5 s to +3.0 s for motor imagery |
| Baseline correction | Subtract pre-stimulus mean from each trial × channel | Pre-stimulus period: tmin to 0 s |
| Epoch rejection | Remove trials exceeding amplitude threshold | Start with 150 µV; aim to keep ≥ 70% of trials |

---

### ➜ Next: Tutorial 3 — EEG Analysis & Classification

In Tutorial 3 we will use these clean epochs to:
- Compute ERPs and compare left vs right hand conditions
- Analyse the frequency spectrum and band power
- Build time-frequency maps showing ERD/ERS
- Extract features and train a left/right classifier
